In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score
from metagen.framework import Domain
from metagen.metaheuristics.rs import RandomSearch

In [2]:
X = pd.read_csv('../dataset/dataset_in.txt', index_col=0).drop(columns=['P2', 'Q2'])
y_columns = [f'V{i}' for i in range(2,15)] + ['PERD', 'QGEN7'] #V1 ES SIEMPRE 1.0 EN DATASET
Y = pd.read_csv('../dataset/dataset_out.txt', index_col=0).loc[:, y_columns]
#FILTRAR DATOS ERRÓNEOS (PÉRDIDAS NEGATIVAS)
mask = Y['PERD'] >= 0
X = X[mask]
Y = Y[mask]

In [3]:
# VALIDACIÓN POR RETENCIÓN PARA EVALUACIÓN FINAL DEL MEJOR MODELO
n_samples = X.shape[0]
train_size = int(n_samples * 0.8)
x_train, y_train = X[:train_size], Y[:train_size]
x_test, y_test = X[train_size:], Y[train_size:]

In [4]:
NN_domain = Domain()
NN_domain.define_categorical("activation", ["relu", "elu"])
NN_domain.define_real("dropout", 0.0, 0.3)
NN_domain.define_categorical("use_batch_norm", [True, False])
NN_domain.define_real("learning_rate", 1e-4, 1e-2)
NN_domain.define_categorical("optimizer", ["sgd", "adam"])
NN_domain.define_integer("n_hidden_q", 1, 3)
NN_domain.define_integer("n_neurons_q", 32, 128)
NN_domain.define_integer("n_hidden_v", 1, 2)
NN_domain.define_integer("n_neurons_v", 64, 256)
NN_domain.define_integer("n_hidden_p", 2, 5)
NN_domain.define_integer("n_neurons_p", 64, 256)

In [5]:
def build_meta_neural_network(solution, input_dim, num_voltages):
    inputs = tf.keras.Input(shape=(input_dim,), name='input_x')
    x_norm = inputs

    activation = solution["activation"]
    dropout_rate = solution["dropout"]
    use_batch_norm = solution["use_batch_norm"]

    if solution["optimizer"] == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=solution["learning_rate"])
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=solution["learning_rate"])

    x1 = x_norm
    for _ in range(solution["n_hidden_q"]):
        x1 = tf.keras.layers.Dense(solution["n_neurons_q"])(x1)
        if use_batch_norm: x1 = tf.keras.layers.BatchNormalization()(x1)
        x1 = tf.keras.layers.Activation(activation)(x1)
        if dropout_rate > 0: x1 = tf.keras.layers.Dropout(dropout_rate)(x1)

    out_qgen = tf.keras.layers.Dense(1, activation='linear', name='out_qgen7')(x1)
    merged = tf.keras.layers.Concatenate()([x_norm, out_qgen])

    x2 = merged
    for _ in range(solution["n_hidden_v"]):
        x2 = tf.keras.layers.Dense(solution["n_neurons_v"])(x2)
        if use_batch_norm: x2 = tf.keras.layers.BatchNormalization()(x2)
        x2 = tf.keras.layers.Activation(activation)(x2)
        if dropout_rate > 0: x2 = tf.keras.layers.Dropout(dropout_rate)(x2)

    out_v = tf.keras.layers.Dense(num_voltages, activation='linear', name='out_volt')(x2)

    x3 = merged
    for _ in range(solution["n_hidden_p"]):
        x3 = tf.keras.layers.Dense(solution["n_neurons_p"])(x3)
        if use_batch_norm: x3 = tf.keras.layers.BatchNormalization()(x3)
        x3 = tf.keras.layers.Activation(activation)(x3)
        if dropout_rate > 0: x3 = tf.keras.layers.Dropout(dropout_rate)(x3)

    out_perd = tf.keras.layers.Dense(1, activation='linear', name='out_perd')(x3)

    model = tf.keras.Model(inputs=inputs, outputs=[out_v, out_perd, out_qgen])

    model.compile(
        optimizer=optimizer,
        loss_weights={'out_qgen7': 1.0, 'out_volt': 1.0, 'out_perd': 2.0},
        loss={'out_qgen7': 'mse', 'out_volt': 'mse', 'out_perd': 'mse'},
        metrics={'out_qgen7': ['mae', 'mape', tf.keras.metrics.R2Score(name='r2_score')],
            'out_volt': ['mae', 'mape', tf.keras.metrics.R2Score(name='r2_score')],
            'out_perd': ['mae', 'mape', tf.keras.metrics.R2Score(name='r2_score')]},
    )
    return model

In [6]:
def build_nn_fitness(num_volts, idx_perd, idx_qgen, idx_volts, x, y):
    def nn_fitness(solution):
        model = build_meta_neural_network(solution, input_dim=x.shape[1], num_voltages=num_volts)
        tscv = TimeSeriesSplit(n_splits=5)

        val_losses = []

        for train_index, test_index in tscv.split(x):
            x_tr, x_val = x.iloc[train_index], x.iloc[test_index]
            y_tr, y_val = y.iloc[train_index], y.iloc[test_index]

            scaler_x_fold = MinMaxScaler()
            scaler_y_fold = MinMaxScaler()

            xs_tr = scaler_x_fold.fit_transform(x_tr)
            ys_tr = scaler_y_fold.fit_transform(y_tr)

            xs_val = scaler_x_fold.transform(x_val)
            ys_val = scaler_y_fold.transform(y_val)

            dict_y_tr = {
                'out_volt': ys_tr[:, idx_volts],
                'out_perd': ys_tr[:, idx_perd:idx_perd+1],
                'out_qgen7': ys_tr[:, idx_qgen:idx_qgen+1]
            }
            dict_y_val = {
                'out_volt': ys_val[:, idx_volts],
                'out_perd': ys_val[:, idx_perd:idx_perd+1],
                'out_qgen7': ys_val[:, idx_qgen:idx_qgen+1]
            }

            stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

            model.fit(xs_tr, dict_y_tr, validation_data=(xs_val, dict_y_val),
                      epochs=15, batch_size=256, verbose=0, callbacks=[stop_early])

            metrics = model.evaluate(xs_val, dict_y_val, verbose=0, return_dict=True)
            val_losses.append(metrics['loss'])

        return np.mean(val_losses)
    return nn_fitness

In [7]:
volts = len([c for c in Y.columns if c.startswith('V')])
idx_v = [Y.columns.get_loc(c) for c in Y.columns if c.startswith('V')]
idx_p = Y.columns.get_loc('PERD')
idx_q = Y.columns.get_loc('QGEN7')

In [8]:
fitness = build_nn_fitness(volts, idx_p, idx_q, idx_v, x_train, y_train)
NN_solution = RandomSearch(NN_domain, fitness, population_size=2, max_iterations=20).run()
print(NN_solution)

F = 0.00877946736291051	{activation = relu , dropout = 0.13057230112028143 , learning_rate = 0.0041609884009389435 , n_hidden_p = 4 , n_hidden_q = 3 , n_hidden_v = 2 , n_neurons_p = 131 , n_neurons_q = 121 , n_neurons_v = 96 , optimizer = adam , use_batch_norm = True}


In [9]:
best_model = build_meta_neural_network(NN_solution, input_dim=x_train.shape[1], num_voltages=volts)

In [10]:
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

xs_train = scaler_x.fit_transform(x_train)
ys_train = scaler_y.fit_transform(y_train)
xs_test = scaler_x.transform(x_test)

dict_y_train = {
    'out_volt': ys_train[:, idx_v],
    'out_perd': ys_train[:, idx_p:idx_p+1],
    'out_qgen7': ys_train[:, idx_q:idx_q+1]
}

In [11]:
stop_early_final = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
best_model.fit(xs_train, dict_y_train, epochs=30, batch_size=256, verbose=1, callbacks=[stop_early_final])

Epoch 1/30
328/328 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.3308 - out_perd_loss: 0.0743 - out_perd_mae: 0.1582 - out_perd_mape: 987.0649 - out_perd_r2_score: -8.7954 - out_qgen7_loss: 0.1269 - out_qgen7_mae: 0.2398 - out_qgen7_mape: 10780.1328 - out_qgen7_r2_score: -2.2914 - out_volt_loss: 0.0551 - out_volt_mae: 0.1466 - out_volt_mape: 3061.7300 - out_volt_r2_score: -2.5833
Epoch 2/30
328/328 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0276 - out_perd_loss: 0.0026 - out_perd_mae: 0.0390 - out_perd_mape: 96.0575 - out_perd_r2_score: 0.6530 - out_qgen7_loss: 0.0174 - out_qgen7_mae: 0.0948 - out_qgen7_mape: 8672.2119 - out_qgen7_r2_score: 0.5498 - out_volt_loss: 0.0050 - out_volt_mae: 0.0536 - out_volt_mape: 2465.5591 - out_volt_r2_score: 0.6827  
Epoch 3/30
328/328 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0162 - out_perd_loss: 9.4320e-04 - out_perd_mae: 0.0234 - out_perd_mape: 492.1649 - out_perd_r2_score: 0.8757 - out_qgen7_loss: 0.0118 - out_qgen7_mae: 0.0730 - out_qgen7_mape: 9651

In [12]:
preds_list = best_model.predict(xs_test)
preds_concat = np.hstack([preds_list[0], preds_list[1], preds_list[2]])
predictions = scaler_y.inverse_transform(preds_concat)

656/656 ━━━━━━━━━━━━━━━━━━━━ 1s 778us/step


In [13]:
random_MAE = mean_absolute_error(y_test, predictions, multioutput='raw_values')
random_MAPE = mean_absolute_percentage_error(y_test, predictions, multioutput='raw_values')

# DESPLAZAMIENTO PARA EVITAR DIVISIÓN POR VALORES CERCANO A 0. CONSECUENCIA: MAPE ALGO OPTIMISTA
random_MAPE_shifted = mean_absolute_percentage_error(y_test + 1.0, predictions + 1.0, multioutput='raw_values')
random_R2 = r2_score(y_test, predictions, multioutput='raw_values')

In [14]:
print("\n--- RESULTADOS CON MAPE DESPLAZADO ---")
random_data_score_shifted = np.vstack([random_MAE, random_MAPE_shifted, random_R2])
evaluations_shifted = pd.DataFrame(random_data_score_shifted, columns=y_columns, index=['MAE', 'MAPE', 'R2'])
display(evaluations_shifted)

print("\n--- RESULTADOS SIN DESPLAZAMIENTO ---")
random_data_score = np.vstack([random_MAE, random_MAPE, random_R2])
evaluations = pd.DataFrame(random_data_score, columns=y_columns, index=['MAE', 'MAPE', 'R2'])
display(evaluations)


--- RESULTADOS CON MAPE DESPLAZADO ---


,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,PERD,QGEN7
MAE,0.002149,0.002154,0.002156,0.002148,0.002154,0.002154,0.002154,0.002155,0.002156,0.002157,0.002157,0.001535,0.001532,0.299521,0.009180
MAPE,0.001088,0.001091,0.001092,0.001088,0.001091,0.001091,0.001091,0.001092,0.001092,0.001092,0.001093,0.000774,0.000773,0.031811,0.008594
R2,0.961254,0.961718,0.961977,0.962241,0.962102,0.962142,0.962082,0.962070,0.962052,0.962044,0.962037,0.975087,0.975182,0.968692,0.843725



--- RESULTADOS SIN DESPLAZAMIENTO ---


,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,PERD,QGEN7
MAE,0.002149,0.002154,0.002156,0.002148,0.002154,0.002154,0.002154,0.002155,0.002156,0.002157,0.002157,0.001535,0.001532,0.299521,9.180139e-03
MAPE,0.002204,0.002210,0.002213,0.002206,0.002211,0.002212,0.002212,0.002212,0.002213,0.002214,0.002214,0.001563,0.001559,0.037285,1.351303e+13
R2,0.961254,0.961718,0.961977,0.962241,0.962102,0.962142,0.962082,0.962070,0.962052,0.962044,0.962037,0.975087,0.975182,0.968692,8.437254e-01


In [15]:
# EVALUACIÓN FINAL CON MEDIA DE VOLTAJES (MAPE DESPLAZADO)
volts_columns = [c for c in y_train.columns if c.startswith('V')]
pd.DataFrame({
    'Voltajes (Desplazados)': evaluations_shifted[volts_columns].mean(axis=1),
    'Pérdidas (Desplazada)': evaluations_shifted['PERD'],
    'Potencia Reactiva (Desplazada)': evaluations_shifted['QGEN7']
})

,Voltajes (Desplazados),Pérdidas (Desplazada),Potencia Reactiva (Desplazada)
MAE,0.002058,0.299521,0.009180
MAPE,0.001042,0.031811,0.008594
R2,0.963999,0.968692,0.843725


In [16]:
# EVALUACIÓN FINAL CON MEDIA DE VOLTAJES (MAPE SIN DESPLAZAMIENTO)
pd.DataFrame({
    'Voltajes': evaluations[volts_columns].mean(axis=1),
    'Pérdidas': evaluations['PERD'],
    'Potencia Reactiva': evaluations['QGEN7']
})

,Voltajes,Pérdidas,Potencia Reactiva
MAE,0.002058,0.299521,9.180139e-03
MAPE,0.002111,0.037285,1.351303e+13
R2,0.963999,0.968692,8.437254e-01
